# 03 — Optimization & Final Benchmarking
**Research Objective:** Produce optimized inference artifacts (ONNX + quantized) and compare models (F1, latency, model size). This notebook treats optimization and benchmarking as reproducible evaluation steps; quantization/export is optional and requires `optimum` + `onnxruntime`.
## Method Overview
**Why:** Optimization improves inference speed and reduces model size for deployment. **Assumptions:** `optimum`, `onnxruntime` and a compatible toolchain are available for quantization.

In [15]:
# Configuration & reproducibility (single action)
ROOT = r"D:\\NLP"
MODELS_MAP = {
    "Model_A_Base": ROOT + "\\models\\model_A_finetuned",
    "Model_B_DAPT": ROOT + "\\models\\model_B_finetuned",
    "Model_C_XLMR": ROOT + "\\models\\model_C_finetuned",
    "Model_D_Opt":  ROOT + "\\models\\model_D_optimized"
}
SEED = 42
try:
    set_seed(SEED)
except NameError:
    import random, numpy as np, torch
    def set_seed(seed: int = SEED, deterministic: bool = False):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    set_seed(SEED)
print('Seed set to', SEED)

Seed set to 42


In [ ]:
# Model size helper (single action)
import os


def get_model_size_mb(path):
    target_files = ["model.safetensors", "pytorch_model.bin", "model_quantized.onnx", "model.onnx"]
    if not os.path.exists(path):
        return None
    for f in os.listdir(path):
        if f in target_files:
            return os.path.getsize(os.path.join(path, f)) / (1024*1024)
    # fallback: sum files in root
    return sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))) / (1024*1024)
print("Model size helper ready.")

Model size helper ready.


In [17]:
# Optional: run optimization (single action) - guarded
RUN_OPTIMIZE = False
if RUN_OPTIMIZE:
    from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    src = MODELS_MAP["Model_B_DAPT"]
    out = MODELS_MAP["Model_D_Opt"]
    os.makedirs(out, exist_ok=True)
    model = ORTModelForSequenceClassification.from_pretrained(src, export=True)
    quantizer = ORTQuantizer.from_pretrained(model)
    cfg = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=out, quantization_config=cfg)
    from transformers import AutoTokenizer
    AutoTokenizer.from_pretrained(src).save_pretrained(out)
    print("Optimization complete; artifacts at", out)
else:
    print("Optimization skipped (RUN_OPTIMIZE=False).")

Optimization skipped (RUN_OPTIMIZE=False).


In [18]:
# Benchmarking (single action) — run a light-weight benchmarking demo (subset) to compute latency & F1
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import f1_score
import numpy as np, time, os, torch

# Try to import ONNX runtime for optimized artifacts
try:
    import onnxruntime as ort
    ort_available = True
except Exception:
    ort_available = False

test_data = load_dataset("ccosme/FiReCS")
# Collapse splits as in original script and sample small subset for quick demo
slices = []
for s in ['train','test','validation']:
    if s in test_data: slices.append(test_data[s])
full = concatenate_datasets(slices)
full = full.map(lambda ex: {"text": ex["review"], "label": (0 if isinstance(ex["label"], (int,float)) else {"negative":0,"positive":1,"neutral":2}.get(ex["label"].lower(),2))})
subset = [full[i] for i in range(min(50, len(full)))]
results = []
for name, path in MODELS_MAP.items():
    if not os.path.exists(path):
        print(f"Skipping {name}: path not found.")
        continue
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        # Some tokenizers (notably recent Mistral ones) need a regex fix flag; attempt it safely
        try:
            tokenizer = AutoTokenizer.from_pretrained(path, fix_mistral_regex=True)
        except TypeError:
            # older versions of tokenizers may not accept the kwarg
            tokenizer = AutoTokenizer.from_pretrained(path)

        # Prefer PyTorch model load, but if missing/unsupported try ONNX if available
        model = None
        onnx_session = None
        try:
            model = AutoModelForSequenceClassification.from_pretrained(path).to("cpu")
        except Exception as e_model:
            # If PyTorch weights not present, try ONNX
            if ort_available:
                onnx_candidates = ["model_quantized.onnx", "model.onnx"]
                onnx_path = None
                for c in onnx_candidates:
                    candidate = os.path.join(path, c)
                    if os.path.exists(candidate):
                        onnx_path = candidate
                        break
                if onnx_path is not None:
                    try:
                        onnx_session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
                        print(f"Using ONNXRuntime for {name} ({onnx_path}).")
                    except Exception as e_onnx:
                        print(f"Skipping {name}: failed to create ONNX session - {e_onnx}")
                        continue
                else:
                    print(f"Skipping {name}: no PyTorch checkpoint and no ONNX artifact found ({e_model}).")
                    continue
            else:
                print(f"Skipping {name}: failed to load model and onnxruntime not available ({e_model}).")
                continue

        preds, labels, latencies = [], [], []
        warmup = True
        for item in subset:
            text = item["text"]
            label = item["label"]

            if model is not None:
                # PyTorch path
                inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
                # Some models (DistilBERT) do not accept token_type_ids — remove if present
                if "token_type_ids" in inputs:
                    inputs.pop("token_type_ids")

                if warmup:
                    with torch.no_grad(): model(**inputs)
                    warmup = False
                t0 = time.time()
                with torch.no_grad(): out = model(**inputs)
                t1 = time.time()
                logits = out.logits
                p = int(torch.argmax(logits, dim=-1))

            else:
                # ONNX path: build numpy inputs and map to session input names
                tok_inputs = tokenizer(text, return_tensors="np", truncation=True, max_length=128)
                # Convert int arrays to expected dtype
                for k,v in list(tok_inputs.items()):
                    tok_inputs[k] = np.asarray(v).astype(np.int64)

                # Remove token_type_ids if not expected by the session
                session_input_names = [i.name for i in onnx_session.get_inputs()]
                if "token_type_ids" in tok_inputs and "token_type_ids" not in session_input_names:
                    tok_inputs.pop("token_type_ids")

                ort_inputs = {name: tok_inputs[name] for name in session_input_names if name in tok_inputs}

                if warmup:
                    onnx_session.run(None, ort_inputs)
                    warmup = False
                t0 = time.time()
                outputs = onnx_session.run(None, ort_inputs)
                t1 = time.time()
                # assume logits are first output
                logits = outputs[0]
                # logits may be shape (1, num_labels)
                p = int(np.argmax(logits, axis=-1).item())

            preds.append(p); labels.append(label); latencies.append((t1 - t0) * 1000.0)

        size = get_model_size_mb(path)
        if len(preds) > 0:
            f1 = f1_score(labels, preds, average='macro')
            latency = np.mean(latencies)
        else:
            f1 = np.nan; latency = np.nan
        results.append({"Model": name, "F1": f1, "Latency_ms": latency, "Size_MB": size if size is not None else np.nan})
    except Exception as e:
        print(f"Failed to benchmark {name}: {e}")
# Display (light) summary
import pandas as pd
if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print('No benchmark results to show; run with available model artifacts.')

The tokenizer you are loading from 'D:\\NLP\models\model_D_optimized' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Using ONNXRuntime for Model_D_Opt (D:\\NLP\models\model_D_optimized\model_quantized.onnx).


,Model,F1,Latency_ms,Size_MB
0,Model_A_Base,0.190476,25.400324,516.243412
1,Model_B_DAPT,0.176471,23.591948,516.243412
2,Model_C_XLMR,0.161616,45.618868,1060.684261
3,Model_D_Opt,0.176471,4.761763,129.453015


## Outputs & Observations
- The notebook performs a conservative benchmarking subset by default. For full experiments, run the benchmark cell on full test splits (as in the original script).  
- Optimization to ONNX and quantization requires `optimum` and `onnxruntime` and is hardware/toolchain-dependent.